# Pokrycie kodu testami czarnoskrzynkowymi — zamiana liczby na polską nazwę

## 1. Poprawki w testach

Testy importowały funkcję z modułu `number_converter`, który na zajęciach 1 był tylko pustą atrapą
(zawsze zwracał `""`). Implementacja zamieszczona na eportalu jest w pliku `number_converter_2.py`, więc jedyna potrzebna zmiana to import.

## 2. Uruchomienie testów z pomiarem pokrycia

Używamy coverage.py przez wtyczkę `pytest-cov`:
- `--cov=number_converter_2` — mierzymy tylko kod testowanej implementacji,
- `--cov-branch` — oprócz linii mierzymy też gałęzie (czy każdy `if` poszedł w obie strony),
- `--cov-context=test` — coverage zapisuje dla każdej linii, **które testy** ją wykonały
  (potrzebne do znalezienia testu, który sam daje największe pokrycie).

Wywołujemy pytesta przez `sys.executable`, żeby użyć tego samego środowiska co kernel notebooka.

In [13]:
import os
import subprocess
import sys
from pathlib import Path

ORYGINAL = Path("../../zajecia_1/zad1/number_converter_tests.py")
POPRAWIONY = Path("number_converter_tests.py")
IMPLEMENTACJA_DIR = Path("../../zajecia_1/zad1").resolve()

env = {**os.environ, "PYTHONPATH": str(IMPLEMENTACJA_DIR), "PYTHONIOENCODING": "utf-8", "PY_COLORS": "0", "NO_COLOR": "1"}
Path(".coverage").unlink(missing_ok=True)

wynik = subprocess.run(
    [sys.executable, "-m", "pytest", str(POPRAWIONY), "-q", "-p", "no:cacheprovider", "--rootdir=.",
     "--tb=no", "--color=yes", "--cov=number_converter_2", "--cov-branch", "--cov-context=test", "--cov-report=term-missing"],
    env=env, capture_output=True, text=True, encoding="utf-8",
)
# pomijamy długie ścieżki z nagłówka raportu, zostawiamy podsumowanie
print(wynik.stdout.replace(str(IMPLEMENTACJA_DIR) + os.sep, ""))

....................FF.........FF..                                      [100%]
=============================== tests coverage ================================
_______________ coverage: platform win32, python 3.12.0-final-0 _______________

Name                                                                                                                    Stmts   Miss Branch BrPart  Cover   Missing
-------------------------------------------------------------------------------------------------------------------------------------------------------------------
number_converter_2.py      37      1     22      1    97%   25
-------------------------------------------------------------------------------------------------------------------------------------------------------------------
TOTAL                                                                                                                      37      1     22      1    97%
=========================== short test summary inf

### Wynik testów

31 z 35 przypadków przechodzi. Te niepowodzenia to wartościowe wyniki:

| Przypadek | Oczekiwaliśmy | Implementacja | Przyczyna |
|---|---|---|---|
| `1000` | `tysiąc` | `jeden tysiąc` | różnica w zapisie — w polszczyźnie standardowo mówimy „tysiąc”, kod dokleja „jeden” |
| `1001` | `tysiąc jeden` | `jeden tysiąc jeden` | ta sama przyczyna |
| `-1` | `minus jeden` | `ValueError` | implementacja obsługuje tylko zakres `<0, 999999>`, my założyliśmy `<-999999, 999999>` |
| `-999999` | `minus dziewięćset …` | `ValueError` | jw. |

Przypadki ujemne wynikają z naszego przeoczenia - założyliśmy inny przedział liczb. Przypadek `1000` to błąd (niezgodność ze zwyczajem językowym) w implementacji.

## 3. Osiągnięte pokrycie kodu

In [12]:
import json

subprocess.run([sys.executable, "-m", "coverage", "json", "-q", "-o", "coverage.json"], env=env, check=True)

raport = json.loads(Path("coverage.json").read_text(encoding="utf-8"))
plik = next(iter(raport["files"]))  # mierzymy tylko jeden plik
s = raport["files"][plik]["summary"]

print(f"Instrukcje (linie): {s['covered_lines']}/{s['num_statements']}  ->  {s['covered_lines'] / s['num_statements']:.1%}")
print(f"Gałęzie:            {s['covered_branches']}/{s['num_branches']}  ->  {s['covered_branches'] / s['num_branches']:.1%}")
print(f"Łącznie (coverage): {s['percent_covered']:.1f}%")
print()
print("Niewykonane linie:  ", raport["files"][plik]["missing_lines"])
print("Niewykonane gałęzie:", [f"{a}->{b}" for a, b in raport["files"][plik]["missing_branches"]])

Instrukcje (linie): 36/37  ->  97.3%
Gałęzie:            21/22  ->  95.5%
Łącznie (coverage): 96.6%

Niewykonane linie:   [25]
Niewykonane gałęzie: ['24->25']


Pokrycie jest bardzo wysokie: **36 z 37 instrukcji (97,3%)** i **21 z 22 gałęzi (95,5%)**, łącznie **96,6%**.

Niewykonana jest jedna linia — `25` — i jedna gałąź, która do niej prowadzi (`24->25`).

## 4. Który przypadek testowy sam daje największe pokrycie?

Uwaga: linia 1 (`def convert_to_polish_name...`) wykonuje się przy imporcie modułu, czyli podczas zbierania testów,
a nie w żadnym teście. Dlatego pojedynczy test może pokryć najwyżej 36 z 37 linii.

In [5]:
import re
import tempfile

import coverage

cov = coverage.Coverage(data_file=".coverage")
cov.load()
dane = cov.get_data()
testy = sorted(c for c in dane.measured_contexts() if c)  # pusty kontekst = import modułu


def pokrycie_testu(kontekst):
    with tempfile.TemporaryDirectory() as tmp:
        out = Path(tmp) / "c.json"
        # kontekst to wyrażenie regularne, a nazwy testów zawierają [ ] - trzeba je escapować
        cov.json_report(outfile=str(out), contexts=["^" + re.escape(kontekst) + "$"])
        return json.loads(out.read_text(encoding="utf-8"))["totals"]


def ladna_nazwa(kontekst):
    # "plik::test[parametry]|run" -> "test[parametry]" z polskimi znakami
    nazwa = kontekst.split("::", 1)[1].removesuffix("|run")
    return nazwa.encode("ascii", "backslashreplace").decode("unicode_escape")


wiersze = []
for kontekst in testy:
    t = pokrycie_testu(kontekst)
    wiersze.append((ladna_nazwa(kontekst), t["covered_lines"], t["num_statements"],
                    t["covered_branches"], t["num_branches"], t["percent_covered"]))
wiersze.sort(key=lambda w: (-w[5], w[0]))

print(f"{'test':<62} {'linie':>7} {'gałęzie':>8} {'łącznie':>8}")
for nazwa, cl, ns, cb, nb, proc in wiersze:
    nazwa = nazwa if len(nazwa) <= 60 else nazwa[:57] + "..."
    print(f"{nazwa:<62} {cl:>3}/{ns:<3} {cb:>3}/{nb:<4} {proc:>7.1f}%")

test                                                             linie  gałęzie  łącznie
test_convert_valid_number[123456-sto dwadzieścia trzy tys...    30/37   11/22      69.5%
test_convert_valid_number[999999-dziewięćset dziewięćdzie...    30/37   11/22      69.5%
test_convert_valid_number[22000-dwadzieścia dwa tysiące]        28/37   11/22      66.1%
test_convert_valid_number[25000-dwadzieścia pięć tysięcy]       28/37   11/22      66.1%
test_convert_valid_number[100000-sto tysięcy]                   27/37   11/22      64.4%
test_convert_valid_number[2000-dwa tysiące]                     27/37   11/22      64.4%
test_convert_valid_number[4000-cztery tysiące]                  27/37   11/22      64.4%
test_convert_valid_number[5000-pięć tysięcy]                    27/37   11/22      64.4%
test_convert_valid_number[1001-tysiąc jeden]                    27/37   10/22      62.7%
test_convert_valid_number[999-dziewięćset dziewięćdziesią...    27/37    9/22      61.0%
test_convert_valid_nu

Najwięcej pokrywają samodzielnie przypadki `123456` i `999999` (remis): 30/37 linii i 11/22 gałęzi, łącznie 69,5%. To liczby sześciocyfrowe, więc przechodzą przez obie części algorytmu (tysiące i reszta), a w każdej z nich wykonują setki, dziesiątki i jedności.

Najmniej dają przypadki spoza zakresu i ujemne (tylko sprawdzenie zakresu i `raise`) oraz `0` (kończy się na `return "zero"` jeszcze przed główną logiką).

Widać też dużą redundancję: wiele przypadków (np. `1`, `2`, `9` albo `100`, `200`, `300`, `400`, `500`) przechodzi dokładnie przez te same linie i gałęzie. Z punktu widzenia pokrycia kodu są równoważne — ale z punktu widzenia testów czarnoskrzynkowych nie są zbędne, bo sprawdzają różne wpisy w tablicach słów (`units`, `hundreds`), których coverage w ogóle nie rozróżnia.

## 5. Linie, które nie zostały wykonane

### Analiza linii 25

```python
def three_digits_to_words(n):
    if n == 0:
        return ""          # linia 25 - nigdy niewykonana
```

Funkcja pomocnicza `three_digits_to_words` jest wywoływana tylko w trzech miejscach:


```python
if thousands > 0:
        if thousands == 1:
            words.append("jeden tysiąc")
        elif thousands % 10 in [2, 3, 4] and not (12 <= thousands % 100 <= 14):
            words.append(three_digits_to_words(thousands) + " tysiące") # tylko gdy thousands > 0
        else:
            words.append(three_digits_to_words(thousands) + " tysięcy") # tylko gdy thousands > 0

    if remainder > 0:
        words.append(three_digits_to_words(remainder)) # tylko gdy remainder > 0
```

Każde wywołanie jest poprzedzone warunkiem, który gwarantuje `n > 0`, a przypadek `number == 0` jest obsłużony
wcześniej (`return "zero"`). Linia 25 jest więc nieosiągalna - nie istnieje żadne wejście `convert_to_polish_name(x)`, które by ją wykonało. To kod martwy (zabezpieczenie „na wszelki wypadek”).

**Jakie przypadki testowe należałoby dodać?** Żaden test czarnoskrzynkowy nie pokryje tej linii. Możliwe rozwiązania:
1. usunąć martwy warunek z implementacji - nie da się go przetestować, a nie zmienia zachowania;
2. jeśli trzeba go zostawić jako zabezpieczenie na przyszłość - oznaczyć `# pragma: no cover`, żeby nie zaniżał raportu;

## 6. Wnioski

1. **Osiągnięte pokrycie**: 97,3% instrukcji (36/37) i 95,5% gałęzi (21/22). Testy pisane czarnoskrzynkowo, tylko na podstawie specyfikacji i podziału na klasy równoważności (cyfry, nastki, dziesiątki, setki, odmiana „tysiąc / tysiące / tysięcy”, wartości brzegowe), pokryły praktycznie cały kod, choć nie znaliśmy implementacji.
2. **Najlepszy pojedynczy przypadek**: `123456` oraz `999999` (remis) - po 69,5% (30/37 linii, 11/22 gałęzi) każdy. Wiele przypadków jest z punktu widzenia coverage redundantna, ale sprawdza różne wartości w tablicach słów, czego coverage nie mierzy.
3. **Niewykonana linia 25** jest nieosiągalna - żaden test jej nie pokryje. Zamiast dopisywać testy, należałoby usunąć ten martwy kod (albo oznaczyć `# pragma: no cover`). Wtedy nasze testy dawałyby 100% pokrycia linii i gałęzi.
4. **Pokrycie ≠ poprawność**: mimo 97% pokrycia 4 testy nie przechodzą. Ujawniły:
   - przeoczony przez nas zakres liczb - implementacja nie obsługuje liczb ujemnych (zakres `<0, 999999>`), a my zakładaliśmy `<-999999, 999999>`,
   - błąd w implementacji - `1000` daje „jeden tysiąc” zamiast „tysiąc”.